# pseudocode 
## The first step of the code will try import the necessary libraries. Establish the variables for Home_modes & Alert_severities>
## We then need to set up the classes for both sensor and alert and a list for each to call on
## Once we have defined each sensor and what the alerts are we then need to establish what scenarios they will be called upon (else/if)
## We then need to create the output for the user to be able to see the situation

"""
HomeGuard Security System Simulator
Author: Chris Fulton
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""

In [32]:
# STEP 1
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

In [33]:
# STEP 2
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold,
        "current_value": None,
        "status": "normal"
    }
    return sensor
def create_alert(severity, message, sensor_id, timestamp):
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

sensors = [
    create_sensor("MO_LIV", "Living Room", "motion"),                    
    create_sensor("TEMP_KIT", "Kitchen", "temperature", threshold=35),      
    create_sensor("DOOR_FRNT", "Front Door", "door"),                       
    create_sensor("SMOKE_BDRM", "Bedroom", "smoke")                         
]

alerts = [
    create_alert("LOW", "Motion detected in Living Room", "MO_LIV", "2023-10-01 10:00:00"),
    create_alert("CRITICAL", "Temperature in Kitchen is high", "TEMP_KIT", "2023-10-01 10:05:00"),
    create_alert("MEDIUMN", "Front Door is open", "DOOR_FRNT", "2023-10-01 10:10:00"),
    create_alert("HIGH", "Smoke detected in Bedroom", "SMOKE_BDRM", "2023-10-01 10:15:00")
]

In [34]:
# STEP 3
def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95:
            return True
            
    elif sensor_type == "motion":
        if reading_value == True:
            return True
            
    elif sensor_type == "door":
        if str(reading_value).upper() == "OPEN":
            return True
            
    elif sensor_type == "smoke":
        if reading_value == True or str(reading_value).upper() == "DETECTED":
            return True
            
    return False


def should_trigger_security_alert(sensor, reading_value, system_mode):
    sensor_type = sensor["type"]
    if sensor_type in ["motion", "door"]:
        if system_mode in ["AWAY", "SLEEP"] and is_abnormal_reading(sensor, reading_value):
            return True
            
    return False

In [35]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [36]:
# STEP 4
def generate_reading(sensor):
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        return random.randint(30, 100)
    elif sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "door":
        return random.choice(["CLOSED", "OPEN"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
    
    return None


def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_id = sensor["id"]
    location = sensor["location"]
    
    sensor["current_value"] = reading_value
    
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        if sensor["type"] == "motion":
            msg = f"Motion detected in {location} while in {system_mode} mode!"
            alerts.append(create_alert("HIGH", msg, sensor_id, timestamp))
        elif sensor["type"] == "door":
            msg = f"Door opened at {location} while system is in {system_mode} mode!"
            alerts.append(create_alert("HIGH", msg, sensor_id, timestamp))

    if sensor["type"] == "temperature":
        if reading_value < 35:
            msg = f"Freezing temperature ({reading_value}°F) detected at {location}! Risk of pipe damage."
            alerts.append(create_alert("CRITICAL", msg, sensor_id, timestamp))
        elif reading_value > 95:
            msg = f"High temperature ({reading_value}°F) detected at {location}! Risk of overheating."
            alerts.append(create_alert("HIGH", msg, sensor_id, timestamp))
            
    elif sensor["type"] == "smoke" and reading_value == "DETECTED":
        msg = f"Smoke detected at {location}! Immediate action required."
        alerts.append(create_alert("CRITICAL", msg, sensor_id, timestamp))
    
    if system_mode == "HOME" and sensor["type"] == "temperature":
        if reading_value < 65 or reading_value > 75:
            msg = f"Comfort notice: Temperature at {location} is {reading_value}°F (outside normal range)."
            alerts.append(create_alert("LOW", msg, sensor_id, timestamp))
            
    return alerts


def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} [{alert['severity']}] ({alert['timestamp']}) - Sensor {alert['sensor_id']}: {alert['message']}")


def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [37]:
# Test
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: False
[ALERT!] 🚨 [HIGH] (20:27:12) - Sensor MO_LIV: Motion detected in Living Room while in AWAY mode!


In [38]:
# STEP 5
class Sensor:
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):

        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
        self.status = "normal"
    
    def read(self):
        if self.type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.type == "motion":
            self.current_value = random.choice([True, False, False])
        elif self.type == "door":
            self.current_value = random.choice(["CLOSED", "CLOSED", "OPEN"])
        elif self.type == "smoke":
            self.current_value = random.choice(["CLEAR", "CLEAR", "CLEAR", "DETECTED"])
            
        return self.current_value
        
    def isAbnormal(self):
        if self.current_value is None:
            return False
            
        if self.type == "temperature":
            low_limit = self.threshold if self.threshold is not None else 35
            if self.current_value < low_limit or self.current_value > 95:
                return True
        elif self.type == "motion":
            if self.current_value == True:
                return True
        elif self.type == "door":
            if str(self.current_value).upper() == "OPEN":
                return True
        elif self.type == "smoke":
            if self.current_value == True or str(self.current_value).upper() == "DETECTED":
                return True
                
        return False
  
    def reset(self):
        self.current_value = None
    
    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
 
sensor_objects = [
    Sensor("MO_LIV", "Living Room", "motion"),
    Sensor("TEMP_KIT", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_FRNT", "Front Door", "door"),
    Sensor("SMOKE_BDRM", "Bedroom", "smoke")
]


In [39]:
# Test your class
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 77
Is abnormal: False
Sensor info: TEST_001 (Test Room): 77


In [40]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading with custom formatting based on type
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation for 3 minutes in AWAY mode
    run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 20:27:12
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 84°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:12
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 [HIGH] (20:27:12) - Sensor MOTION_001: Motion detected in Living Room while in AWAY mode!
[LOG] [20:27:12] Sending notification to homeowner...
[READING] Kitchen Temperature: 41°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:13
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 91°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!


In [41]:
def run_simulation(duration_minutes=5, system_mode="HOME"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading with custom formatting based on type
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation for 12 minutes in SLEEP mode
    run_simulation(duration_minutes=12, system_mode="SLEEP")

=== HomeGuard Security System ===
Mode: SLEEP


Time: 20:27:13
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 [HIGH] (20:27:13) - Sensor MOTION_001: Motion detected in Living Room while in SLEEP mode!
[LOG] [20:27:13] Sending notification to homeowner...
[READING] Kitchen Temperature: 63°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:14
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 [HIGH] (20:27:14) - Sensor MOTION_001: Motion detected in Living Room while in SLEEP mode!
[LOG] [20:27:14] Sending notification to homeowner...
[READING] Kitchen Temperature: 88°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:14
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 42°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 [HIGH] (20:27:14) - Sensor DOOR_001: Door opened at Front Door while system is in SLEEP mode!
[LOG] [20:27:14] Sending notification to homeowner...
[READING] Bedroom Smoke: C

In [42]:
def run_simulation(duration_minutes=5, system_mode="HOME"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading with custom formatting based on type
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation for 9 minutes in HOME mode
    run_simulation(duration_minutes=9, system_mode="HOME")

=== HomeGuard Security System ===
Mode: HOME


Time: 20:27:19
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 93°F (Abnormal)
[ALERT!] ℹ️ [LOW] (20:27:19) - Sensor TEMP_001: Comfort notice: Temperature at Kitchen is 93°F (outside normal range).
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:20
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 87°F (Abnormal)
[ALERT!] ℹ️ [LOW] (20:27:20) - Sensor TEMP_001: Comfort notice: Temperature at Kitchen is 87°F (outside normal range).
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:20
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 83°F (Abnormal)
[ALERT!] ℹ️ [LOW] (20:27:20) - Sensor TEMP_001: Comfort notice: Temperature at Kitchen is 83°F (outside normal range).
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 20:27:21
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 64°F (Abnormal)
